# V64 - Nested stacking with E5

Pre-registered gap: v61 stacked four sparse views but E5 existed only as a concat candidate in v58, never inside a learned combination. This version nests word-only, E5 dense, assessment, and mentions under the same leak-proof protocol with a fresh seed. E5 loads from the v58 cache, a dataset path, or encodes on the fly. One gate, same locks.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 4040
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v64")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Load views including cached E5

Identical preprocessing to the hunt standard. E5 never touches labels before the gate.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada".split())
MN = np.zeros((len(all_users), len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    keys = set(re.findall(r"[a-z]+", (str(row["module_name"]) + " " + str(row["description_and_syllabus"])).lower())) - STOP if row is not None else set()
    keys = {k for k in keys if len(k) > 3}
    for i, d in enumerate(low):
        MN[i, j] = sum(d.count(k) for k in keys)
MN = (MN - MN[:n_train].mean(0)) / (MN[:n_train].std(0) + 1e-9)
E = None
for p in [Path("results/v58/e5_users.npy"), Path("/kaggle/input/intelligo-e5/e5_users.npy"), OUT / "e5_users.npy"]:
    if p.exists():
        E = np.load(p)
        print("E5 cache:", p)
        break
if E is None:
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("intfloat/multilingual-e5-base")
    E = np.asarray(enc.encode(["query: " + d for d in docs], batch_size=32, show_progress_bar=True, normalize_embeddings=True))
    np.save(OUT / "e5_users.npy", E)
VIEWS = {"word": W, "e5": E, "assess": A, "mentions": np.nan_to_num(MN)}
print("views:", {k: v.shape for k, v in VIEWS.items()})

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def view_oof(V, idx, seed):
    oof = np.zeros((len(idx), len(MODS)))
    skf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    yy, gg = Y[idx], fold_groups[idx]
    st = yy.argmax(1)
    for tr, va in skf.split(V[idx], st, gg):
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(V[idx][tr], yy[tr, j])
            oof[va, j] = m.predict(V[idx][va])
    return oof

names = list(VIEWS)
oof_stack = np.zeros((n_train, len(MODS)))
fids = np.zeros(n_train, int)
outer = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(outer.split(W[:n_train], strata, fold_groups)):
    L1 = np.stack([view_oof(V[:n_train], tr, SEED + fi) for V in VIEWS.values()], axis=-1)
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(L1[:, j, :], Y[tr, j])
        full = []
        for V in VIEWS.values():
            mm = Ridge(alpha=1.0)
            mm.fit(V[:n_train][tr], Y[tr, j])
            full.append(mm.predict(V[:n_train][va]))
        oof_stack[va, j] = m.predict(np.stack(full, axis=-1).reshape(len(va), -1))
    fids[va] = fi + 1
    print(f"outer {fi + 1} done", flush=True)

oof_w = np.zeros_like(Y)
for fi in range(1, N_SPLITS + 1):
    tr, va = np.where(fids != fi)[0], np.where(fids == fi)[0]
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[:n_train][tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[:n_train][va])
base = ndcg5(Y, oof_w)
d = ndcg5(Y, oof_stack) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_stack[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"stack+E5 OOF={ndcg5(Y, oof_stack).mean():.5f} baseline={base.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT stack+E5" if (lo > GATE_FLOOR and wf >= 0) else "KEEP baseline; E5 adds nothing even stacked")

E5 cache: results/v58/e5_users.npy
views: {'word': (5000, 319), 'e5': (5000, 768), 'assess': (5000, 10), 'mentions': (5000, 17)}


outer 1 done


outer 2 done


outer 3 done


outer 4 done


outer 5 done


stack+E5 OOF=0.61948 baseline=0.62732 delta=-0.00783 CI95=[-0.01094,-0.00472] worst-fold=-0.03553
KEEP baseline; E5 adds nothing even stacked
